# Lab 1: Vectorless RAG


## Setup

### Install Dependencies

In [ ]:
!pip install -q pageindex openai python-dotenv pymupdf

### Import Libraries


In [ ]:
import os
import json
import re
import fitz  # PyMuPDF — extracts text from PDF
from openai import OpenAI
from dotenv import load_dotenv

### Load API Keys

In [ ]:
load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# If keys are missing, prompt the user to enter them
if not PAGEINDEX_API_KEY:
    PAGEINDEX_API_KEY = input("Enter your PageIndex API key (get one at https://pageindex.ai): ").strip()
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

print("Keys loaded.")

### Set up the LLM

In [ ]:
def call_llm(prompt, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    """Call a language model via OpenRouter.

    Args:
        prompt: The text prompt to send to the model.
        model: The model identifier on OpenRouter.

    Returns:
        The model's text response.
    """
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )
    msgs = [{"role": "user", "content": prompt}]
    resp = client.chat.completions.create(model=model, messages=msgs, temperature=0, max_tokens=1024)
    return resp.choices[0].message.content.strip()

---
## Step 1 — Extract Text from PDF

### Define PDF Path

In [ ]:
PDF_PATH = "data/CCS 3.31.25 Earnings Release 8-K Exhibit 99.1.pdf"

### Extract Text Function


In [ ]:
def extract_page_text(pdf_path):
    """Extract text from each PDF page.

    Returns:
        dict mapping 1-based page number to extracted text
    """
    doc = fitz.open(pdf_path)
    texts = {}
    for i in range(len(doc)):
        texts[i+1] = doc.load_page(i).get_text()
    doc.close()
    return texts

### Run Extraction

In [ ]:
page_texts = extract_page_text(PDF_PATH)
print(f"Extracted text from {len(page_texts)} pages.")

---
## Step 2 — Build Document Tree (with caching)


### Set Up Caching


In [ ]:
from pageindex import PageIndexClient
from pageindex import utils
import time

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

### Get Cache Path


In [ ]:
def get_cache_path(pdf_path):
    """Generate a cache file path based on the PDF filename."""
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    return os.path.join(CACHE_DIR, f"{pdf_name}_tree.json")

### Load Cached Tree


In [ ]:
def load_cached_tree(pdf_path):
    """Load tree from cache if it exists, otherwise return None."""
    cache_path = get_cache_path(pdf_path)
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            tree = json.load(f)
        print(f"Loaded tree from cache: {cache_path}")
        return tree
    return None

### Save Tree to Cache

In [ ]:
def save_tree_to_cache(pdf_path, tree):
    """Save the PageIndex tree to a JSON file for future reuse."""
    cache_path = get_cache_path(pdf_path)
    with open(cache_path, "w") as f:
        json.dump(tree, f, indent=2)
    print(f"Tree saved to cache: {cache_path}")

### Load or Build the Tree


In [ ]:
pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)

# Try loading from cache first — avoids re-processing the same PDF
tree = load_cached_tree(PDF_PATH)

if tree is None:
    # Cache miss — submit PDF to PageIndex for tree generation
    print("Cache miss — submitting to PageIndex...")
    result = pi.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"Submitted: {doc_id}")

    # Poll until PageIndex finishes processing (max 5 min)
    print("Waiting for PageIndex to process...")
    elapsed = 0
    while elapsed < 300:
        if pi.is_retrieval_ready(doc_id):
            break
        time.sleep(5)
        elapsed += 5
        print(f"  {elapsed}s...")
    else:
        raise TimeoutError(f"PageIndex did not finish within {elapsed}s. Try again later.")

    print(f"Ready after {elapsed}s")
    tree = pi.get_tree(doc_id, node_summary=True)["result"]
    save_tree_to_cache(PDF_PATH, tree)

# Display the tree structure (titles + summaries, no full text)
utils.print_tree(tree, exclude_fields=["text"])

---
## Step 3 — Ask a Question

In [ ]:
QUERY = "What was the total revenue reported in the earnings release?"

---
## Step 4 — LLM Finds Relevant Sections

### Search the Tree


In [ ]:
# Remove full text from tree — the LLM only needs titles + summaries
# to decide which sections are relevant. This keeps the prompt small.
tree_slim = utils.remove_fields(tree.copy(), fields=["text"])

# Ask the LLM to reason over the tree and pick relevant nodes
search_prompt = f"""
You are given a question and a document tree.
Each node has: node_id, title, summary.
Find all nodes likely to contain the answer.

Question: {QUERY}

Document tree:
{json.dumps(tree_slim, indent=2)}

Reply in this JSON format ONLY:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

raw_result = call_llm(search_prompt)
print(raw_result)

### Parse JSON Response

In [ ]:
def parse_json(text):
    """Extract JSON from LLM response, handling markdown code fences."""
    text = re.sub(r"```json\s*|\s*```", "", text.strip())
    s, e = text.find("{"), text.rfind("}")
    if s != -1 and e != -1:
        text = text[s:e+1]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    thinking = ""
    m_think = re.search(r'"thinking"\s*:\s*"((?:[^"\\]|\\.)*)"', text, re.DOTALL)
    if m_think:
        thinking = m_think.group(1)
    node_list = []
    m_nodes = re.search(r'"node_list"\s*:\s*\[(.*?)\]', text, re.DOTALL)
    if m_nodes:
        node_list = re.findall(r'"(\d+)"', m_nodes.group(1))
    return {"thinking": thinking, "node_list": node_list}

### Display Retrieved Nodes


In [ ]:
result = parse_json(raw_result)

# Build a mapping from node_id -> node info (title, page range, etc.)
node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=len(page_texts))

print("Reasoning:", result["thinking"], "\n")
print("Retrieved nodes:")
for nid in result["node_list"]:
    info = node_map[nid]
    pages = info['start_index'] if info['start_index'] == info['end_index'] else f"{info['start_index']}-{info['end_index']}"
    print(f"  {nid} | Pages {pages} | {info['node']['title']}")

---
## Step 5 — LLM Answers from Extracted Text

### Extract Text for Retrieved Nodes

In [ ]:
def text_for_nodes(nodes, node_map, page_texts):
    """Collect extracted text from pages covered by the given nodes."""
    texts, seen = [], set()
    for nid in nodes:
        info = node_map[nid]
        for p in range(info["start_index"], info["end_index"] + 1):
            if p not in seen and p in page_texts:
                texts.append(f"--- Page {p} ---\n{page_texts[p]}")
                seen.add(p)
    return "\n\n".join(texts)

### Build Context


In [ ]:
context = text_for_nodes(result["node_list"], node_map, page_texts)
print(f"Using {len(context.splitlines())} lines of text.")

### Generate Answer


In [ ]:
# Send the extracted text + question to the LLM for the final answer.
answer_prompt = f"""
Answer the question based on the provided text.

Context:
{context}

Question: {QUERY}

Rules:
- Answer only from the context
- If the answer isn't there, say so
- Be concise
"""

answer = call_llm(answer_prompt)
print(answer)

---
## Try It Yourself

Change `QUERY` above and re-run from **Step 4**.